In [28]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [29]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from housepriceprediction.config import DATA_RAW, RANDOM_STATE
from housepriceprediction.preprocessing import build_preprocessor

train_df = pd.read_csv(DATA_RAW / "train.csv", keep_default_na=False, na_values=["NA"])
train_df = train_df[train_df["GrLivArea"] <= 4000].reset_index(drop=True)
y = np.log1p(train_df["SalePrice"])
X = train_df.drop(columns=["Id", "SalePrice"])

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = {}
alpha_hyperparameters = [300, 500, 1000, 2000, 5000]

In [30]:
def evaluate(name, pipeline):
    rmse_scores = -cross_val_score(pipeline, X, y, cv=cv, scoring="neg_root_mean_squared_error")
    results[name] = rmse_scores
    print(f"--- {name} ---")
    print(f"The 5 RMSE scores (log): {np.round(rmse_scores, 4)}")
    print(f"Mean RMSE: {rmse_scores.mean():.4f}")
    print(f"Standard Deviation: {rmse_scores.std():.4f}")

In [31]:
baseline_pipeline = Pipeline([
    ("preprocessor", build_preprocessor(add_derived=True)),
    ("regressor", DummyRegressor(strategy="median")),
])
evaluate("Baseline (median)", baseline_pipeline)

--- Baseline (median) ---
The 5 RMSE scores (log): [0.3977 0.4023 0.4024 0.4184 0.3642]
Mean RMSE: 0.3970
Standard Deviation: 0.0178


In [32]:
for alpha in alpha_hyperparameters:
    ridge_pipeline = Pipeline([
        ("preprocessor", build_preprocessor(add_derived=True)),
        ("scaler", StandardScaler()),
        ("regressor", Ridge(alpha=alpha)),
    ])
    evaluate(f"Ridge (alpha={alpha})", ridge_pipeline)

--- Ridge (alpha=300) ---
The 5 RMSE scores (log): [0.1241 0.1148 0.1158 0.1147 0.1035]
Mean RMSE: 0.1146
Standard Deviation: 0.0066
--- Ridge (alpha=500) ---
The 5 RMSE scores (log): [0.1259 0.1154 0.1159 0.1175 0.1032]
Mean RMSE: 0.1156
Standard Deviation: 0.0072
--- Ridge (alpha=1000) ---
The 5 RMSE scores (log): [0.1309 0.1184 0.1186 0.1248 0.1062]
Mean RMSE: 0.1198
Standard Deviation: 0.0082
--- Ridge (alpha=2000) ---
The 5 RMSE scores (log): [0.1408 0.1262 0.1263 0.138  0.115 ]
Mean RMSE: 0.1293
Standard Deviation: 0.0093
--- Ridge (alpha=5000) ---
The 5 RMSE scores (log): [0.1669 0.1515 0.1512 0.1693 0.1398]
Mean RMSE: 0.1558
Standard Deviation: 0.0110


In [33]:
# Ridge - choosing alpha (5-fold CV, RMSE on log prices)
#
# Grid 1: [0.1, 1, 3, 10, 30, 100, 300] -> error kept decreasing, minimum at the edge (300),
#         so the grid was extended upwards.
# Grid 2: [300, 500, 1000, 2000, 5000]  -> error rises again after 300 (U-shaped curve).
#
# Chosen: alpha = 300, Mean RMSE = 0.1146 (std 0.0066).
#   - The curve is flat between 100 and 500 (differences ~0.001, well below the
#     fold-to-fold std of ~0.006), so a finer grid would only fit noise.
#   - Small alpha (0.1-10): almost no change; the model is slightly unstable but not bad.
#   - Large alpha (1000+): the error grows quickly -> underfitting (coefficients pushed towards 0).
#   - Why a strong penalty helps: 200+ columns after one-hot, many of them correlated
#     (TotalSF / GrLivArea / TotalBsmtSF, GarageCars / GarageArea).
#
# Gain from tuning: 0.1182 (alpha=10) -> 0.1146 (alpha=300), ~3%.
# Baseline (median) for comparison: 0.3970.